# CryptoML-SBox — Re-Run Apple-to-Apple XGBoost pada Dataset Bersama (Jobdesk Mahdi)

Notebook ini siap dijalankan di sesi **Kaggle** atau **Google Colab** baru dari awal (tidak bergantung sesi lama).

**Yang dilakukan notebook ini:**
1. Mengkloning branch `main` terbaru `MahdiHbb1/cryptoml-sbox` (sudah berisi modul Mahdi, Yafi, dan Naila).
2. Menjalankan seluruh unit test `pytest` (verifikasi 53/53 test termasuk validasi AES Standar & K44).
3. Membangun ulang dataset bersama persis seperti Yafi & Naila (`data/sbox_dataset.csv`: `750 chaotic_swap + 750 aes_affine`, `seed=42`, memakai `src/sbox_metrics.py` milik Naila), lalu melatih & melakukan tuning **XGBoost** pada dataset tersebut agar perbandingan **XGBoost vs Random Forest vs MLP** 100% *apple-to-apple*.
4. Menguji enkripsi & dekripsi *lossless* citra **Mandrill** (`data/mandril_color.jpg`) pada mode `CBC2`.
5. Mengekspor `results/sbox_xgb.npy` dan `results/xgb_metrics.json` agar langsung bisa diunduh dan di-commit ke GitHub.

In [ ]:
# Sel 1: Clone repo terbaru dari branch main & install dependensi
import os, subprocess

REPO_URL = "https://github.com/MahdiHbb1/cryptoml-sbox.git"
BRANCH = "main"

if not os.path.exists("cryptoml-sbox"):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL], check=True)
else:
    subprocess.run(["git", "-C", "cryptoml-sbox", "pull", "origin", BRANCH], check=True)

%cd cryptoml-sbox
!pip install -q -r requirements.txt

In [ ]:
# Sel 2: Jalankan seluruh unit test tim (Mahdi + Yafi + Naila)
!python -m pytest -v

In [ ]:
# Sel 3: Bangun dataset bersama persis seperti Yafi & Naila, lalu latih + tuning XGBoost (apple-to-apple)
!python -m src.dataset_builder --n-chaotic 750 --n-aes 750 --seed 42

import json
from pathlib import Path
import numpy as np
from src.aes_affine_generator import build_k44_sbox, build_standard_aes_sbox
from src.sbox_metrics import evaluate_sbox
from src.train_xgboost import compute_composite_score, run_xgboost_pipeline

summary = run_xgboost_pipeline(
    dataset_path="data/sbox_dataset.csv",
    out_sbox_path="results/sbox_xgb.npy",
    out_metrics_path="results/xgb_metrics.json",
    n_screen_candidates=2000,   # Sama persis dengan RF (Yafi) & MLP (Naila)
    top_k=25,                   # Sama persis dengan RF & MLP
    tune=True,                  # Aktifkan RandomizedSearchCV
    seed=42,
    pool_mode="mixed",          # 50% AES-affine + 50% Non-AES chaotic (seimbang seperti RF & MLP)
)

print("=== PERBANDINGAN 3 MODEL ML PADA DATASET BERSAMA (n=1500, seed=42) ===")
for label, path in [
    ("XGBoost (Mahdi)", Path("results/xgb_metrics.json")),
    ("Random Forest (Yafi)", Path("results/rf_metrics.json")),
    ("MLP (Naila)", Path("results/mlp_metrics.json")),
]:
    if path.exists():
        d = json.loads(path.read_text(encoding="utf-8"))
        ml = d["ml_metrics"]
        print(
            f"{label:22s} | R2={ml['r2']:.4f} | MAE={ml['mae']:.4f} | "
            f"RMSE={ml['rmse']:.4f} | Train={ml['train_time_sec']:.2f}s | "
            f"Origin={d['dataset_origin']}"
        )
print("Best XGBoost params:", summary["ml_metrics"]["best_params"])
print("Selected poly/const:", summary.get("selected_sbox_poly"), summary.get("selected_sbox_constant"))

print("\n=== PERBANDINGAN METRIK 1: SELURUH 5 S-BOX ===")
for label, sb in [
    ("AES Standar", build_standard_aes_sbox()),
    ("K44 Alamsyah", build_k44_sbox()),
    ("S-Box XGBoost (Mahdi)", np.load("results/sbox_xgb.npy")),
    ("S-Box RF (Yafi)", np.load("results/sbox_rf.npy")),
    ("S-Box MLP (Naila)", np.load("results/sbox_mlp.npy")),
]:
    m = evaluate_sbox(sb)
    sc = compute_composite_score(m)
    print(
        f"{label:24s} | Bij={m['bijective']} | NL={m['nl']} | "
        f"SAC={m['sac_mean']:.5f} (max={m['sac_max_err']:.5f}) | "
        f"BIC-NL={m['bic_nl']} | BIC-SAC={m['bic_sac']:.5f} | "
        f"DU={m['du']} | FP={m['fixed_points']} | Score={sc:.4f}"
    )

In [ ]:
# Sel 4: Uji enkripsi & dekripsi lossless pada citra Mandrill memakai S-Box XGBoost terbaru
import matplotlib.pyplot as plt
from PIL import Image
from src.aes_cipher_runner import decrypt_image_array, encrypt_image_array

def shannon_entropy(arr):
    p = np.bincount(arr.ravel(), minlength=256) / arr.size
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

sbox_xgb = np.load("results/sbox_xgb.npy")
plain = np.array(Image.open("data/mandril_color.jpg").convert("RGB"), dtype=np.uint8)
plain_flip = plain.copy()
plain_flip[-1, -1, -1] ^= np.uint8(1)  # Ubah 1 bit di piksel terakhir

key = b"CryptoMLSBox2026"
iv = b"MandrillIV123456"

c1 = encrypt_image_array(plain, sbox=sbox_xgb, key=key, iv=iv, mode="CBC2")
c2 = encrypt_image_array(plain_flip, sbox=sbox_xgb, key=key, iv=iv, mode="CBC2")
rec = decrypt_image_array(c1, sbox=sbox_xgb, key=key, iv=iv, orig_shape=plain.shape, mode="CBC2")

npcr = float(np.mean(c1 != c2) * 100.0)
uaci = float(np.mean(np.abs(c1.astype(np.int32) - c2.astype(np.int32)) / 255.0) * 100.0)

print(f"Lossless decrypt : {np.array_equal(rec, plain)}")
print(f"Entropy Plain    : {shannon_entropy(plain):.5f}")
print(f"Entropy Cipher   : {shannon_entropy(c1):.5f}")
print(f"NPCR (C1 vs C2)  : {npcr:.4f}%")
print(f"UACI (C1 vs C2)  : {uaci:.4f}%")

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, img_data, title in zip(axes, [plain, c1, rec], ["Mandrill Asli", "Terenkripsi (AES-CBC2 + S-Box XGBoost)", "Hasil Dekripsi"]):
    ax.imshow(img_data)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Sel 5: Unduh hasil dengan deteksi aman untuk Kaggle vs Google Colab
import os, shutil, sys
from pathlib import Path

is_kaggle = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle/working").exists()
is_colab = ("COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules) and not is_kaggle

if is_kaggle:
    from IPython.display import FileLink, display
    for fname in ["sbox_xgb.npy", "xgb_metrics.json"]:
        src = Path("results") / fname
        dst = Path("/kaggle/working") / fname
        shutil.copy2(src, dst)
        display(FileLink(f"results/{fname}"))
    print("Selesai! Klik link di atas atau ambil dari panel kanan Kaggle: Output -> /kaggle/working/")
elif is_colab:
    from google.colab import files
    files.download("results/sbox_xgb.npy")
    files.download("results/xgb_metrics.json")
else:
    print("Lingkungan lokal — file tersimpan di 'results/sbox_xgb.npy' dan 'results/xgb_metrics.json'.")